# 🏗️ 00 — SETU-Rail Setup: Catalog, Schemas, Volume

**Dhara module — Foundation cell**

Creates the Unity Catalog structure that every other notebook will use:
- Catalog: `setu_rail`
- Schemas: `bronze`, `silver`, `gold`
- Volume: `setu_rail.bronze.raw_files`

Then auto-copies every file from the repo's `raw_data/` folder into the Volume.

**Run on:** Serverless compute. **Safe to re-run.**

## Step 1 — Create catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS setu_rail
COMMENT 'SETU-Rail: AI copilot for Indian Railways (Bharat Bricks Hacks 2026, IIT Madras)';

USE CATALOG setu_rail;

## Step 2 — Create medallion schemas (Bronze/Silver/Gold)

This is the **meaningful Delta Lake usage** judges will look for — not just one schema, a full medallion hierarchy.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS setu_rail.bronze
COMMENT 'Raw, append-only data — as delivered from sources';

CREATE SCHEMA IF NOT EXISTS setu_rail.silver
COMMENT 'Cleaned, joined, enriched data';

CREATE SCHEMA IF NOT EXISTS setu_rail.gold
COMMENT 'ML-ready and analytics-ready curated tables + views';

## Step 3 — Create Volume for raw files

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS setu_rail.bronze.raw_files
COMMENT 'Landing zone for raw CSVs and PDFs (train timings, air quality, Railway Rules, Railways Act)';

## Step 4 — Auto-copy everything from `raw_data/` into the Volume

Re-runnable. Existing files get overwritten.

In [0]:
import os, shutil

# Find the repo root (this notebook lives at <repo>/01_dhara_data_engineering/00_setup...)
notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))
raw_data_dir = os.path.join(repo_root, "raw_data")
volume_path = "/Volumes/setu_rail/bronze/raw_files"

print(f"📂 Repo root:     {repo_root}")
print(f"📂 Raw data dir:  {raw_data_dir}")
print(f"📂 Volume target: {volume_path}")
print()

if not os.path.isdir(raw_data_dir):
    raise FileNotFoundError(
        f"Could not find raw_data/ directory at {raw_data_dir}. "
        "Make sure you imported the repo as a Git folder."
    )

copied = 0
for name in sorted(os.listdir(raw_data_dir)):
    src = os.path.join(raw_data_dir, name)
    if not os.path.isfile(src):
        continue
    dst = os.path.join(volume_path, name)
    shutil.copyfile(src, dst)
    print(f"  ✅ copied {name} ({os.path.getsize(dst):,} bytes)")
    copied += 1

print(f"\n🎉 Done. {copied} files copied into the Volume.")

## Step 5 — Verify

In [0]:
files = dbutils.fs.ls("/Volumes/setu_rail/bronze/raw_files")
print("Files in Volume:\n")
for f in files:
    print(f"  {f.name:<50} {f.size:>12,} bytes")

✅ **Next:** Open `01_ingest_train_timings.ipynb`